In [1]:
# ================================
# 1️⃣ Install
# ================================
!pip install -q transformers datasets

# ================================
# 2️⃣ Imports
# ================================
import json
from tqdm import tqdm
from transformers import pipeline
from datasets import Dataset

# ================================
# 3️⃣ Load Dataset
# ================================
file_path = "/content/OS_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# ================================
# 4️⃣ Labels
# ================================
LABELS = sorted(list(set([ex["output"].strip() for ex in data])))
print("Labels:", LABELS)

# ================================
# 5️⃣ Zero-Shot Pipeline
# ================================
classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",  # DistilBERT NLI
    device=0  # GPU (use -1 for CPU)
)

# ================================
# 6️⃣ Run Zero-Shot Prediction
# ================================
y_true = []
y_pred = []
results = []

for ex in tqdm(dataset):
    text = ex["input"]
    true_label = ex["output"].strip()

    output = classifier(
        text,
        candidate_labels=LABELS,
        multi_label=False
    )

    pred_label = output["labels"][0]

    y_true.append(true_label)
    y_pred.append(pred_label)

    results.append({
        "input": text,
        "ground_truth": true_label,
        "predicted": pred_label,
        "scores": dict(zip(output["labels"], output["scores"]))
    })

# ================================
# 7️⃣ Metrics
# ================================
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n===== ZERO-SHOT RESULTS (DistilBERT) =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ================================
# 8️⃣ Save + Download
# ================================
output_file = "/content/distilbert_zeroshot_predictions.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\n✅ Predictions saved to: {output_file}")

from google.colab import files
files.download(output_file)

Labels: ['based_on', 'implements', 'improves', 'no_relation', 'part_of', 'used_for']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

100%|██████████| 1957/1957 [01:05<00:00, 29.96it/s]



===== ZERO-SHOT RESULTS (DistilBERT) =====
Accuracy : 0.0915
Precision: 0.3880
Recall   : 0.0915
F1 Score : 0.0735

✅ Predictions saved to: /content/distilbert_zeroshot_predictions.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>